In [ ]:
import os
import pandas as pd
import time
from src.models.mc import hybridMonteCarlo
from src.models.longstaff import LSMC_Numpy, LSMC_OpenCL
import src.models.benchmarks as bm
from src.models.pso import PSO_Numpy, PSO_OpenCL
from src.models.utils import checkOpenCL
import argparse

/Users/xiaohuzhang/.pyenv/versions/3.12.9/lib/python3.12/site-packages/pytools/persistent_dict.py:52: RecommendedHashNotFoundWarning: Unable to import recommended hash 'siphash24.siphash13', falling back to 'hashlib.sha256'. Run 'python3 -m pip install siphash24' to install the recommended hash.
  warn("Unable to import recommended hash 'siphash24.siphash13', "


In [15]:
T = 60 / 365
nPath = 20000
nPeriod = 100
nFish = 500

In [ ]:
S0 = 73.67961833	
r = 1.02 / 100  # Convert % to decimal
sigma = 0.284488
K = 73.168224	
opttype = 'p'
opttype=opttype.upper()
mc = hybridMonteCarlo(S0, r, sigma, T, nPath, nPeriod, K, opttype, nFish)
binomial = bm.binomialAmericanOption(S0, K, r, sigma, nPeriod, T, opttype)
lsmc_np = LSMC_Numpy(mc)
lsmc_val_np = float(lsmc_np.longstaff_schwartz_itm_path_fast()[0])
lsmc_cl = LSMC_OpenCL(mc)
lsmc_val_cl = float(lsmc_cl.longstaff_schwartz_itm_path_fast_hybrid()[0])
pso_np = PSO_Numpy(mc, nFish, mc.costPsoAmerOption_np)
pso_val_np = pso_np.solvePsoAmerOption_np()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import sys

# --- File Loading ---

# Define the path to your CSV file.
# Make sure this CSV file is in the same directory as the Python script.
file_path = 'Data/sep2008_5000Fish.xlsx - Sheet1.csv'

try:
    # Read the data from the specified CSV file into a pandas DataFrame
    df = pd.read_csv(file_path)
except FileNotFoundError:
    # Handle the case where the file does not exist
    print(f"Error: The file '{file_path}' was not found.")
    print("Please make sure the CSV file is in the same directory as this script.")
    sys.exit() # Exit the script if the file isn't found

# --- Data Aggregation and Preparation ---

# The data contains multiple entries for the same number of 'days'.
# To create a clear line chart, we should aggregate the data for each day.
# Here, we'll calculate the average 'Diff' value for each day.
# We select only the columns we need for the aggregation.
columns_to_aggregate = [
    'days',
    'binomialDiff',
    'lsmc_cpuDiff',
    'lsmc_gpuDiff',
    'pso_cpuDiff',
    'pso_gpuDiff'
]
df_agg = df[columns_to_aggregate].groupby('days').mean().reset_index()


# --- Visualization ---

# Set up the plot style for better aesthetics
plt.style.use('seaborn-v0_8-whitegrid')

# Create a figure and axes for the plot
fig, ax = plt.subplots(figsize=(14, 8))

# List of the 'Diff' columns we want to plot
diff_columns = [
    'binomialDiff',
    'lsmc_cpuDiff',
    'lsmc_gpuDiff',
    'pso_cpuDiff',
    'pso_gpuDiff'
]

# Plot each 'Diff' column from the aggregated data against 'days'
for column in diff_columns:
    ax.plot(df_agg['days'], df_agg[column], marker='o', linestyle='-', label=column)

# --- Formatting the Chart ---

# Add a title and labels for clarity
ax.set_title('Comparison of Average Model Differences Over Time', fontsize=16, pad=20)
ax.set_xlabel('Days to Maturity', fontsize=12)
ax.set_ylabel('Average Difference Value', fontsize=12)

# Add a legend to identify the lines
ax.legend(title='Difference Models')

# Add grid lines for easier reading
ax.grid(True)

# Ensure the layout is tight to prevent labels from overlapping
plt.tight_layout()

# Display the plot
plt.show()
